# 05 · Diffusion on toy data

Image generators such as Stable Diffusion are **diffusion models**. The idea fits into one notebook if the "images" are points in the plane:

1. **Forward process:** take a data point and add Gaussian noise, a little at a time, until nothing but noise is left.
2. **Training:** teach a small neural network to look at a noisy point and say which noise was added.
3. **Sampling:** start from pure noise and remove the predicted noise step by step. Points appear where the data was.

Everything here is NumPy: the network, its gradients (backpropagation by hand) and the optimizer. Nothing is downloaded except NumPy. This is the method of the DDPM paper (Ho, Jain, Abbeel, 2020) on 2-D data.

Training takes a few seconds; the notebook prints how long.

In [ ]:
import time
import numpy as np
import lab

IMAGE_MODEL = "black-forest-labs/FLUX.1-schnell"   # only for the optional last cell (hosted by Hugging Face)

T = 100                 # number of noise steps
HIDDEN = 64             # width of the network's hidden layers
TRAIN_STEPS = 4000
BATCH = 256
rng = np.random.default_rng(0)

## 1. The data

Two interleaved half circles ("two moons"), 2000 points, shifted and scaled so that both coordinates have mean about 0 and standard deviation about 1. The helper `scatter` draws point clouds side by side as an SVG picture.

In [ ]:
def make_data(n, rng):
    """n points on two interleaved half circles, with a little noise, standardised."""
    half = n // 2
    a = rng.uniform(0, np.pi, half)
    b = rng.uniform(0, np.pi, n - half)
    upper = np.stack([np.cos(a), np.sin(a)], axis=1)
    lower = np.stack([1 - np.cos(b), 0.5 - np.sin(b)], axis=1)
    points = np.concatenate([upper, lower]) + 0.06 * rng.standard_normal((n, 2))
    return (points - np.array([0.5, 0.25])) / np.array([0.87, 0.5])


def scatter(panels, size=190, limit=3.2, max_points=600):
    """Draw point clouds side by side. panels: list of (title, array of shape (n, 2))."""
    parts = []
    for index, (title, points) in enumerate(panels):
        x0 = index * (size + 10)
        dots = "".join(
            f'<circle cx="{x0 + size / 2 + x / limit * size / 2:.1f}" cy="{20 + size / 2 - y / limit * size / 2:.1f}" r="1.3"/>'
            for x, y in points[:max_points]
            if abs(x) < limit and abs(y) < limit
        )
        parts.append(
            f'<rect x="{x0}" y="20" width="{size}" height="{size}" fill="none" stroke="#8a8f98"/>'
            f'<text x="{x0 + size / 2}" y="14" font-size="12" text-anchor="middle" fill="currentColor">{title}</text>'
            f'<g fill="#2553c7" fill-opacity="0.75">{dots}</g>'
        )
    width = len(panels) * (size + 10) - 10
    lab.html(f'<svg viewBox="0 0 {width} {size + 22}" width="{width}" height="{size + 22}" role="img" aria-label="Scatter plots">{"".join(parts)}</svg>')


data = make_data(2000, rng)
print("data:", data.shape, " mean:", data.mean(axis=0).round(2), " std:", data.std(axis=0).round(2))
scatter([("data", data)])

## 2. The forward process: adding noise

Choose small numbers $\beta_1 < \dots < \beta_T$ (the **noise schedule**). One step shrinks the point a little and adds a little noise:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\big(\sqrt{1 - \beta_t}\; x_{t-1},\; \beta_t I\big)$$

With $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \alpha_1 \alpha_2 \cdots \alpha_t$, many steps collapse into one. This is what makes training cheap: we can jump from the data to any step $t$ directly.

$$q(x_t \mid x_0) = \mathcal{N}\big(\sqrt{\bar\alpha_t}\; x_0,\; (1 - \bar\alpha_t) I\big) \qquad\Longleftrightarrow\qquad x_t = \sqrt{\bar\alpha_t}\; x_0 + \sqrt{1 - \bar\alpha_t}\; \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

The schedule is chosen so that $\bar\alpha_T \approx 0$: at the end, $x_T$ is (almost) pure standard Gaussian noise, whatever the data was.

In the code, arrays start at index 0: `betas[0]` is $\beta_1$ and `alpha_bar[t]` is $\bar\alpha_{t+1}$.

In [ ]:
betas = np.linspace(1e-4, 0.1, T)         # beta_1 ... beta_T
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)            # alpha_bar[t] = alphas[0] * ... * alphas[t]


def add_noise(x0, t, eps):
    """Jump from clean points x0 (n, 2) straight to step t (n integers from 0 to T-1) using the noise eps (n, 2)."""
    return np.sqrt(alpha_bar[t])[:, None] * x0 + np.sqrt(1.0 - alpha_bar[t])[:, None] * eps


print("alpha_bar after the first step:", round(float(alpha_bar[0]), 4))
print("alpha_bar after the last step: ", round(float(alpha_bar[-1]), 4))

panels = []
for step in (0, 9, 29, 59, 99):
    t = np.full(len(data), step)
    panels.append((f"t = {step + 1}", add_noise(data, t, rng.standard_normal(data.shape))))
scatter(panels, size=150)

In [ ]:
assert np.all(betas > 0) and np.all(np.diff(betas) > 0)
assert np.isclose(alpha_bar[2], (1 - betas[0]) * (1 - betas[1]) * (1 - betas[2]))
assert alpha_bar[-1] < 0.01, "at the last step almost nothing of the data is left"

# Doing the small steps one after the other gives the same distribution as the direct jump:
# for one starting point, both must have mean sqrt(alpha_bar) * x0 and variance 1 - alpha_bar.
x0 = np.tile([[1.0, -0.5]], (20000, 1))
x = x0.copy()
for step in range(30):
    x = np.sqrt(1.0 - betas[step]) * x + np.sqrt(betas[step]) * rng.standard_normal(x.shape)
assert np.allclose(x.mean(axis=0), np.sqrt(alpha_bar[29]) * x0[0], atol=0.03)
assert np.allclose(x.var(axis=0), 1.0 - alpha_bar[29], atol=0.03)
jump = add_noise(x0, np.full(len(x0), 29), rng.standard_normal(x0.shape))
assert np.allclose(jump.mean(axis=0), x.mean(axis=0), atol=0.03) and np.allclose(jump.var(axis=0), x.var(axis=0), atol=0.03)
print("The forward process is correct.")

## 3. The network and its gradients

The network $\varepsilon_\theta(x_t, t)$ gets a noisy point and the step number and predicts the noise that is in the point. It is a small multilayer perceptron:

- **Input** (18 numbers): the 2 coordinates and 16 numbers that encode $t$ as sines and cosines of different frequencies. (A single number $t$ would be hard for the network to use; this is the same trick as the positional encoding of transformers.)
- **Two hidden layers** with `HIDDEN` units and ReLU: $h_1 = \mathrm{relu}(u W_1 + b_1)$, $h_2 = \mathrm{relu}(h_1 W_2 + b_2)$.
- **Output** (2 numbers): $\hat\varepsilon = h_2 W_3 + b_3$.

The loss is the mean squared error between the true and the predicted noise:

$$L(\theta) = \mathbb{E}_{x_0,\, t,\, \varepsilon}\; \big\lVert \varepsilon - \varepsilon_\theta(x_t, t) \big\rVert^2$$

**Backpropagation** applies the chain rule from the loss back to every weight. For a batch of $n$ points, with $\delta_3 = \partial L / \partial \hat\varepsilon = \tfrac{2}{n} (\hat\varepsilon - \varepsilon)$:

$$\frac{\partial L}{\partial W_3} = h_2^\top \delta_3, \qquad \delta_2 = (\delta_3 W_3^\top) \odot [z_2 > 0], \qquad \frac{\partial L}{\partial W_2} = h_1^\top \delta_2, \qquad \delta_1 = (\delta_2 W_2^\top) \odot [z_1 > 0], \qquad \frac{\partial L}{\partial W_1} = u^\top \delta_1$$

where $z_1, z_2$ are the values before the ReLU, $\odot$ multiplies entry by entry, and the gradient of each bias is the column sum of its $\delta$.

In [ ]:
def time_embedding(t, dim=16):
    """Step numbers (n,) -> (n, dim): sines and cosines with periods from about 6 to about 2600 steps."""
    half = dim // 2
    frequencies = np.exp(-np.log(1000.0) * np.arange(half) / half)
    angles = np.asarray(t, dtype=np.float64)[:, None] * frequencies[None, :]
    return np.concatenate([np.sin(angles), np.cos(angles)], axis=1)


def init_network(rng, hidden=HIDDEN):
    def layer(n_in, n_out):
        return rng.standard_normal((n_in, n_out)) * np.sqrt(2.0 / n_in), np.zeros(n_out)

    W1, b1 = layer(2 + 16, hidden)
    W2, b2 = layer(hidden, hidden)
    W3, b3 = layer(hidden, 2)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": 0.1 * W3, "b3": b3}


def predict_noise(params, x, t):
    """The forward pass. Returns the prediction (n, 2) and the intermediate values that backpropagation needs."""
    u = np.concatenate([x, time_embedding(t)], axis=1)       # (n, 18)
    z1 = u @ params["W1"] + params["b1"]                      # (n, hidden)
    h1 = np.maximum(z1, 0.0)
    z2 = h1 @ params["W2"] + params["b2"]                     # (n, hidden)
    h2 = np.maximum(z2, 0.0)
    out = h2 @ params["W3"] + params["b3"]                    # (n, 2)
    return out, (u, z1, h1, z2, h2)


def loss_and_gradients(params, x_t, t, eps):
    """Mean squared error between eps and the prediction, and its gradient for every parameter."""
    out, (u, z1, h1, z2, h2) = predict_noise(params, x_t, t)
    diff = out - eps
    loss = np.mean(np.sum(diff ** 2, axis=1))
    d3 = 2.0 * diff / len(x_t)                                # dL/d(out)
    d2 = (d3 @ params["W3"].T) * (z2 > 0)                     # dL/d(z2)
    d1 = (d2 @ params["W2"].T) * (z1 > 0)                     # dL/d(z1)
    gradients = {
        "W3": h2.T @ d3, "b3": d3.sum(axis=0),
        "W2": h1.T @ d2, "b2": d2.sum(axis=0),
        "W1": u.T @ d1, "b1": d1.sum(axis=0),
    }
    return loss, gradients

Hand-written gradients are easy to get wrong, so we test them. The **gradient check** compares each formula with a numerical derivative: nudge one weight by $\pm h$ and see how the loss changes,

$$\frac{\partial L}{\partial w} \approx \frac{L(w + h) - L(w - h)}{2h}.$$

In [ ]:
check_rng = np.random.default_rng(1)
check_params = init_network(check_rng, hidden=8)
cx, ct, ceps = check_rng.standard_normal((5, 2)), check_rng.integers(0, T, 5), check_rng.standard_normal((5, 2))
_, analytic = loss_and_gradients(check_params, cx, ct, ceps)

h = 1e-6
worst = 0.0
for name, values in check_params.items():
    for index in list(np.ndindex(values.shape))[:: max(1, values.size // 12)]:      # a dozen entries per parameter
        original = values[index]
        values[index] = original + h
        loss_plus, _ = loss_and_gradients(check_params, cx, ct, ceps)
        values[index] = original - h
        loss_minus, _ = loss_and_gradients(check_params, cx, ct, ceps)
        values[index] = original
        numeric = (loss_plus - loss_minus) / (2 * h)
        worst = max(worst, abs(numeric - analytic[name][index]))
print("largest difference between the formula and the numerical derivative:", worst)
assert worst < 1e-6
print("The gradients are correct.")

## 4. Training

One training step:

1. take a batch of data points $x_0$,
2. pick a random step $t$ for each point and random noise $\varepsilon$,
3. compute the noisy points $x_t$ with `add_noise`,
4. let the network predict $\varepsilon$ from $(x_t, t)$ and move the weights against the gradient of the loss.

The weights are updated with **Adam**: gradient descent with a running average of the gradient ($m$) and of its square ($v$), so that every weight gets its own step size. The loss cannot reach 0: at large $t$ the point is almost pure noise and the network can recover nearly all of it, but at small $t$ the added noise is tiny and hidden under the spread of the data, so it can only be guessed. What matters is that the loss falls well below 2, the value for a network that always predicts 0 (the noise has 2 coordinates with variance 1 each).

In [ ]:
params = init_network(rng)
m = {name: np.zeros_like(value) for name, value in params.items()}
v = {name: np.zeros_like(value) for name, value in params.items()}
LEARNING_RATE, BETA1, BETA2, EPSILON = 2e-3, 0.9, 0.999, 1e-8

losses = []
start = time.time()
for step in range(1, TRAIN_STEPS + 1):
    x0 = data[rng.integers(0, len(data), BATCH)]
    t = rng.integers(0, T, BATCH)
    eps = rng.standard_normal((BATCH, 2))
    loss, gradients = loss_and_gradients(params, add_noise(x0, t, eps), t, eps)
    losses.append(loss)

    # Adam. The learning rate falls linearly to zero over the last 30 % of the steps.
    rate = LEARNING_RATE * min(1.0, (TRAIN_STEPS - step) / (0.3 * TRAIN_STEPS))
    for name in params:
        m[name] = BETA1 * m[name] + (1 - BETA1) * gradients[name]
        v[name] = BETA2 * v[name] + (1 - BETA2) * gradients[name] ** 2
        m_hat = m[name] / (1 - BETA1 ** step)
        v_hat = v[name] / (1 - BETA2 ** step)
        params[name] -= rate * m_hat / (np.sqrt(v_hat) + EPSILON)

    if step % 1000 == 0:
        print(f"step {step:5d}   loss (average of the last 1000 steps): {np.mean(losses[-1000:]):.3f}")

train_seconds = time.time() - start
print(f"trained {TRAIN_STEPS} steps in {train_seconds:.1f} s")

In [ ]:
def loss_curve(losses, width=600, height=160, window=50):
    """The loss, averaged over windows of steps, as an SVG line."""
    averaged = np.asarray(losses)[: len(losses) // window * window].reshape(-1, window).mean(axis=1)
    top = max(2.0, float(averaged.max()))
    points = " ".join(f"{i / (len(averaged) - 1) * width:.1f},{height - value / top * height:.1f}" for i, value in enumerate(averaged))
    lab.html(
        f'<svg viewBox="0 0 {width} {height + 18}" width="{width}" height="{height + 18}" role="img" aria-label="Training loss">'
        f'<rect width="{width}" height="{height}" fill="none" stroke="#8a8f98"/>'
        f'<polyline points="{points}" fill="none" stroke="#2553c7" stroke-width="1.5"/>'
        f'<text x="4" y="{height + 14}" font-size="12" fill="currentColor">training loss, from 0 (bottom) to {top:.1f} (top), over {len(losses)} steps</text></svg>'
    )


loss_curve(losses)

## 5. Sampling: from noise to data

Now run the process backwards. Start with $x_T \sim \mathcal{N}(0, I)$ and, for $t = T, T-1, \dots, 1$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\; \varepsilon_\theta(x_t, t) \right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I), \quad \sigma_t^2 = \beta_t$$

The bracket removes the predicted noise of this step; $\sigma_t z$ puts a little fresh noise back in (except in the very last step, where $z = 0$). Without the fresh noise, the samples collapse towards the middle of the data.

In [ ]:
def sample(params, n, rng, keep=()):
    """Generate n points. Also returns the intermediate clouds at the steps listed in keep."""
    x = rng.standard_normal((n, 2))                              # x_T: pure noise
    kept = {}
    for step in range(T - 1, -1, -1):                            # step = t - 1, from T - 1 down to 0
        eps_hat, _ = predict_noise(params, x, np.full(n, step))
        mean = (x - betas[step] / np.sqrt(1.0 - alpha_bar[step]) * eps_hat) / np.sqrt(alphas[step])
        noise = rng.standard_normal((n, 2)) if step > 0 else 0.0
        x = mean + np.sqrt(betas[step]) * noise
        if step in keep:
            kept[step] = x.copy()
    return x, kept


start = time.time()
samples, kept = sample(params, 1000, rng, keep=(60, 30, 10))
print(f"1000 samples in {time.time() - start:.2f} s")

noise = rng.standard_normal((1000, 2))
scatter([("start: noise", noise), ("after step 61", kept[60]), ("after step 31", kept[30]), ("after step 11", kept[10]), ("result", samples)], size=150)
scatter([("generated", samples), ("real data", data[:1000])])

The generated points were never in the data set. The network only learned in which *direction* the data lies from any noisy point, for every noise level.

How good are the samples? A simple measure: for every generated point, the distance to the nearest real data point. For comparison, the same for plain Gaussian noise.

In [ ]:
def nearest_distance(points, reference):
    """For each point, the distance to its nearest neighbour in reference."""
    differences = points[:, None, :] - reference[None, :, :]
    return np.sqrt((differences ** 2).sum(axis=-1)).min(axis=1)


reference = make_data(4000, np.random.default_rng(1))
generated_distance = nearest_distance(samples, reference).mean()
noise_distance = nearest_distance(noise, reference).mean()
print(f"mean distance to the nearest data point: generated {generated_distance:.3f}, plain noise {noise_distance:.3f}")

assert np.mean(losses[-500:]) < 0.85 * np.mean(losses[:20]), "the loss should fall during training"
assert generated_distance < 0.5 * noise_distance, "generated points should lie much closer to the data than noise does"
assert np.all(np.abs(samples.mean(axis=0) - data.mean(axis=0)) < 0.2), "the samples should be centred like the data"
print("The model generates points that look like the data.")

## From points to pictures

A text-to-image model is the same recipe at another scale: the "point" is an image (or a compressed version of it) with many thousands of coordinates, the network is a large U-Net or transformer instead of our perceptron with about 5500 weights, and the prompt is an extra input, like $t$ here.

**Optional:** such a model does not fit into a browser tab, but you can call a hosted one. Press **HF token** at the top of the page, paste your Hugging Face token, and run the next cell.

In [ ]:
if lab.hf_token():
    picture = await lab.inference.text_to_image(IMAGE_MODEL, "two crescent moons made of blue dots on white paper, scientific plot")
    lab.show(picture)
else:
    print("No Hugging Face token is set, so this optional cell was skipped.")

## Questions

1. Set `TRAIN_STEPS = 500` and run again from the top. What do the samples look like, and what does the check say?
2. In `sample`, remove the fresh noise (set `noise = 0.0` in every step). Describe the result and explain it.
3. Change `T` to 20 and run again from the top. Which check fails first, and why does that matter for sampling, which starts from pure noise?
4. The loss weights all steps $t$ equally. Which steps do you think are the hardest for the network, and why?